In [ ]:
# [1/5] Mount Google Drive & Setup Path
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

print('Project root: /content/drive/MyDrive/code')

In [ ]:
# [2/5] Install Dependencies
!pip install -q z3-solver 2>/dev/null

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

from src.prepare_k10 import prepare_k10_data

print('Ready.')

In [ ]:
# [3/5] CONFIG — change these

# Model: "Qwen4b" | "Qwen8b" | "Ministral8b"
MODEL = "Qwen8b"

# Dataset: "val" | "test" | "test_folio" | "test_willow"
DATASET = "val"

# Z3 timeout per candidate (ms)
Z3_TIMEOUT_MS = 5_000

# Train/val split ratio
TRAIN_RATIO = 0.9

# Random seed for split
SEED = 42

print(f"Model:      {MODEL}")
print(f"Dataset:    {DATASET}")
print(f"Z3 timeout: {Z3_TIMEOUT_MS}ms")
print(f"Train ratio: {TRAIN_RATIO}")

In [ ]:
# [4/5] Step 1 — Dedup + Z3 Filter  (no intermediate file saved)
#
# Reads raw k10, deduplicates per sentence, keeps only Z3-parsable candidates.
# Data is passed directly to Step 2 in memory (save_to_disk=False).

k10_data = prepare_k10_data(
    model=MODEL,
    dataset=DATASET,
    z3_timeout_ms=Z3_TIMEOUT_MS,
    save_to_disk=False,
)["data"]

In [ ]:
# [5/5] Step 2 — Build Selector Training Data
#
# Uses k10_data (already deduped + Z3-filtered in memory from Step 1).
# Computes Z3 LE labels on filtered k10 candidates, builds (NL, FOL, label)
# triples from GT (label=1) + k10 LE=1 (label=1) + k10 LE=0 (label=0).
# No perturbations. Split by sentence_id (9:1).
#
# Output: data/results/{MODEL}/k10/{MODEL}_k10_{DATASET}_selector_train.json
#         data/results/{MODEL}/k10/{MODEL}_k10_{DATASET}_selector_val.json

import json
import random
import time
from collections import Counter
from pathlib import Path

import sys
sys.path.insert(0, '/content/drive/MyDrive/code')

_ROOT = Path('/content/drive/MyDrive/code')
DATA_DIR = _ROOT / "data"
RESULTS_DIR = DATA_DIR / "results" / MODEL / "k10"

# ---- paths ----
gt_path = DATA_DIR / f"{DATASET}.json"
_suffix = DATASET.replace("test_", "").replace("test", "")
_tag = f"{MODEL}_k10_{_suffix}" if _suffix else f"{MODEL}_k10"

train_out = RESULTS_DIR / f"{_tag}_selector_train.json"
val_out = RESULTS_DIR / f"{_tag}_selector_val.json"

print("=" * 55)
print("  SELECTOR TRAINING DATA (M4)")
print(f"  Model:   {MODEL}")
print(f"  Dataset: {DATASET}")
print(f"  GT:      {gt_path}")
print(f"  Output:  {train_out.name}  /  {val_out.name}")
print("=" * 55)

# ---- load GT (k10_data already in memory from Step 1) ----
with open(gt_path, encoding='utf-8') as f:
    gt_data = json.load(f)

gt_map = {i: item["FOL"] for i, item in enumerate(gt_data)}
n_sentences = len(k10_data)
n_cands = sum(len(e.get("candidates", [])) for e in k10_data)
print(f"\n  In-memory k10: {n_sentences} sentences, {n_cands} filtered candidates")

# ---- compute Z3 LE labels ----
print(f"\n  Computing Z3 LE labels (timeout={Z3_TIMEOUT_MS}ms)...")
from src.eval.z3_equiv import check_equivalence

t0 = time.perf_counter()
total_cands = n_cands
done = 0

for entry in k10_data:
    sid = entry.get("sentence_id", -1)
    gt = gt_map.get(sid, "")
    for cand in entry.get("candidates", []):
        fol = cand.get("fol", "")
        if fol and gt:
            try:
                eq = check_equivalence(
                    fol, gt, sentence_id=sid,
                    candidate_id=str(cand.get("candidate_id", "")),
                    timeout_ms=Z3_TIMEOUT_MS,
                )
                cand["z3_le"] = 1 if eq.is_equivalent else 0
            except Exception:
                cand["z3_le"] = None
        else:
            cand["z3_le"] = None
        done += 1

    if done % 500 == 0:
        elapsed = time.perf_counter() - t0
        print(f"    [{done}/{total_cands}]  elapsed={elapsed:.1f}s")

elapsed = time.perf_counter() - t0
print(f"  Done in {elapsed:.1f}s")

# ---- extract samples ----
print(f"\n  Extracting samples...")

samples: list = []
seen_fols: set = set()

def _fol_hash(fol: str) -> str:
    import hashlib
    return hashlib.md5((fol or "").strip().encode()).hexdigest()[:16]

# GT as positive (label=1)
for i, item in enumerate(gt_data):
    fol = item.get("FOL", "")
    if fol:
        h = _fol_hash(fol)
        seen_fols.add(h)
        samples.append({
            "nl": item.get("NL", ""),
            "fol": fol,
            "label": 1,
            "source": "gt",
            "sentence_id": i,
        })

# k10 candidates
for entry in k10_data:
    sid = entry.get("sentence_id", -1)
    nl = entry.get("nl", "")
    for cand in entry.get("candidates", []):
        fol = cand.get("fol", "")
        le = cand.get("z3_le")
        if not fol or le is None:
            continue
        h = _fol_hash(fol)
        if h in seen_fols:
            continue
        seen_fols.add(h)
        samples.append({
            "nl": nl,
            "fol": fol,
            "label": le,
            "source": "k10",
            "sentence_id": sid,
        })

# ---- split by sentence_id (9:1) ----
rng = random.Random(SEED)
all_sids = sorted(set(s["sentence_id"] for s in samples))
rng.shuffle(all_sids)
n_train_sids = int(TRAIN_RATIO * len(all_sids))
train_sids = set(all_sids[:n_train_sids])
val_sids = set(all_sids[n_train_sids:])

train_samples = [s for s in samples if s["sentence_id"] in train_sids]
val_samples = [s for s in samples if s["sentence_id"] in val_sids]

# ---- save ----
def _save(samples_list, out_path, name):
    lc = Counter(s["label"] for s in samples_list)
    sc = Counter(s["source"] for s in samples_list)
    payload = {
        "description": f"M4 selector training data — {name} split",
        "model": MODEL,
        "dataset": DATASET,
        "n_positive": lc.get(1, 0),
        "n_negative": lc.get(0, 0),
        "n_total": len(samples_list),
        "source_breakdown": dict(sc.most_common()),
        "samples": samples_list,
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    size_kb = out_path.stat().st_size / 1024
    print(f"  Saved: {out_path}  ({len(samples_list)} samples, {size_kb:.1f} KB)")
    return lc

print()
tc = _save(train_samples, train_out, "train")
vc = _save(val_samples, val_out, "val")

# ---- stats ----
print(f"\n  {'─' * 40}")
print(f"  Total sentences:  {len(all_sids)}")
print(f"  Train sentences:  {len(train_sids)}")
print(f"  Val sentences:    {len(val_sids)}")
print(f"  Train samples:    {len(train_samples)}  (pos={tc.get(1,0)}, neg={tc.get(0,0)})")
print(f"  Val samples:      {len(val_samples)}  (pos={vc.get(1,0)}, neg={vc.get(0,0)})")
print(f"\n  Done — ready for M4 selector training.")
print("=" * 55)